# Ingesta Bronze nacional — cinco fuentes

La libreta ejecuta la ingesta desde **VS Code Remote-SSH conectado al primario EMR**, utilizando el kernel **American Tower - PySpark**. El código reside en el repositorio del primario y se distribuye automáticamente a los diez trabajadores.

El contrato incluye Ookla móvil en los 29 trimestres 2019Q1–2026Q1, ambos conjuntos OSIPTEL, los tres originales INEI y WorldPop Perú 2020. Se conservan archivos originales sin filtrar ni transformar. La escritura se limita a **Bronze y Operations**; el bucket Artifacts permanece sin modificaciones.

Cada ejecución crea una corrida nueva. La ingesta se realiza únicamente al ejecutar las celdas de esta libreta. No se requieren comandos de carga ni una ejecución local.

**Configuración para el primario EMR**

El archivo `bronze_config.completed.json` contiene los buckets configurados y las fuentes institucionales. Las rutas relativas de originales se resuelven desde la raíz del repositorio remoto. La entrada XLSX de OSIPTEL utiliza el original auditado `datasets/osiptel/archivo_estaciones.xlsx`, porque su enlace SharePoint devolvió HTTP 403. Su SHA-256 se valida antes de incorporarlo; la fecha original de descarga desconocida permanece declarada.

`AT_BRONZE_CONFIG` permite seleccionar otro JSON. Por defecto no se reanuda ningún plan y se ejecuta la ingesta completa. Un cambio deliberado de `EXECUTE_INGESTION` a `False` permite inspeccionar exclusivamente el inventario.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "at_pipeline" / "runtime.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("El repositorio completo debe estar disponible en el nodo primario.")
sys.path.insert(0, str(REPO_ROOT))

from at_pipeline.contracts import Settings

CONFIG_PATH = Path(os.environ.get(
    "AT_BRONZE_CONFIG", str(REPO_ROOT / "notebooks" / "bronze_config.completed.json")
))
if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Falta bronze_config.completed.json en el repositorio remoto.")
configuration = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
for source in ("osiptel_cobertura", "osiptel_estaciones"):
    locator = configuration["inputs"][source]
    if "://" not in locator and not Path(locator).is_absolute():
        configuration["inputs"][source] = str(REPO_ROOT / locator)
settings = Settings.from_dict(configuration)
RESUME_PLAN_URI = None  # Una corrida nueva; solo se cambia para reanudar un plan compatible.
EXECUTE_INGESTION = True
print({"periodos": len(settings.periods), "particiones": settings.transfer_partitions})

**Sesión Spark y distribución del paquete**

Se solicitan diez executors, cada uno con cuatro núcleos, 9 GiB de memoria JVM y 2 GiB adicionales. Una transferencia reserva cuatro núcleos por executor y dispone de cuatro conexiones de transferencia. El ApplicationMaster reserva 1 GiB en total para que YARN pueda alojar los diez executors.

El paquete se registra mediante `addPyFile` y su ZIP permanece disponible durante toda la sesión. La copia verificable del código y los eventos Spark se conservan en Operations. La asignación dinámica y la ejecución especulativa quedan deshabilitadas. Un kernel nuevo permite aplicar la configuración sin reutilizar una sesión anterior.

In [ ]:
import uuid
from datetime import datetime, timezone

from at_pipeline.observability import configure_logging, logger, persist_logs
from at_pipeline.runtime import BronzeRun, create_spark, distribute_code
from at_pipeline.storage import Storage

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
LOG_PATH = configure_logging(RUN_ID, settings.cache_root)
logger.info("Registro operativo habilitado antes de iniciar Spark")
storage = Storage(settings.region)
spark = create_spark(settings)

package = distribute_code(REPO_ROOT / "at_pipeline", settings, storage, spark, RUN_ID)
code_sha256 = package["code_sha256"]
run = BronzeRun(settings, spark, storage, RUN_ID, code_sha256)
print({"run_id": RUN_ID, "application_id": spark.sparkContext.applicationId,
       "code_sha256": code_sha256, "archive_path": package["archive_path"],
       "log_path": str(LOG_PATH)})

**Verificación de la plataforma observada**

Se verifica ejecución en el primario EMR, release `emr-7.1.0`, Spark 3.5.0 y once nodos `m5.xlarge`: un primario y dos núcleos On-Demand, junto con ocho tareas Spot. Se comprueban acceso y región de Bronze y Operations.

Una prueba simultánea con barrera verifica diez hosts diferentes, las dependencias y la escritura/lectura desde cada trabajador. Los fallos devuelven su excepción concreta al driver. La caché prueba escritura real y utiliza los temporales autorizados de YARN si la ruta preferida carece de permisos.

In [ ]:
try:
    environment = run.preflight()
    print({"cluster_id": environment["cluster_id"], "topologia": environment["topology"],
           "hosts_verificados": len({sample["host"] for sample in environment["worker_samples"]})})
    print([{"host": sample["host"], "cache_root": sample["cache_root"],
            "free_bytes": sample["free_bytes"]} for sample in environment["worker_samples"]])
finally:
    persist_logs(storage, settings, RUN_ID, LOG_PATH)

**Planificación o reanudación**

La ejecución inicial construye un inventario nuevo. Se descubren exclusivamente los Parquet Ookla `type=mobile` y se preserva el listado original de cada trimestre. Se registran las dos entradas OSIPTEL, las tres capas INEI y el GeoTIFF WorldPop. Los originales locales se incorporan mediante esta celda a una zona de preparación de Operations, conservando su hash.

Se congelan tamaños e identificadores del proveedor y se comprueba que el inventario contiene las cinco fuentes, los 29 trimestres y las tres capas INEI. Bronze conserva los RAR originales; la extracción y validación de las entidades geográficas corresponden a Silver. El plan se conserva en Operations antes de transferir los datasets.

In [ ]:
try:
    plan = run.load_plan(RESUME_PLAN_URI) if RESUME_PLAN_URI else run.build_plan()
    print({"plan_id": plan["plan_id"], "objetos": len(plan["entries"]),
           "volumen_GiB": round(plan["total_bytes"] / 1024**3, 3),
           "limite_GiB": settings.max_total_bytes / 1024**3,
           "plan_uri": f"s3://{settings.buckets['operations']}/evidence/run_id={plan['run_id']}/plan.json"})
except Exception as exc:
    run.fail(type(exc).__name__)
    raise
finally:
    persist_logs(storage, settings, RUN_ID, LOG_PATH)

**Transferencia de originales y conciliación final**

Ookla se transfiere mediante Spark en diez particiones, mientras el primario transfiere las fuentes institucionales de forma concurrente. WorldPop no bloquea el inicio de las transferencias Ookla. Las descargas mantienen memoria acotada, calculan SHA-256 y validan el contenedor; la copia S3 se vuelve a leer para verificar íntegramente su tamaño y hash.

Los mensajes muestran las fases, actividad cada 15 segundos, tareas Spark, recibos por objeto y checkpoints. Los logs se conservan en Operations incluso al fallar una celda. Los rechazos bloquean la publicación de éxito.

Se concilian el inventario y todos los recibos mediante Python y Spark antes de publicar `manifest.json`, `_SUCCESS` por fuente y el marcador final de corrida. El `bundle_uri` resultante constituye la entrada para Silver.

In [ ]:
try:
    if EXECUTE_INGESTION:
        receipts = run.execute()
        summary = run.publish()
        BUNDLE_URI = summary["bundle_uri"]
        result_path = REPO_ROOT / "outputs" / "latest_bronze.json"
        result_path.parent.mkdir(exist_ok=True)
        result_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        print(json.dumps(summary, ensure_ascii=False, indent=2))
    else:
        logger.info("PLAN_ONLY: plan persistido; transferencia de datasets pendiente de ejecución")
finally:
    persist_logs(storage, settings, RUN_ID, LOG_PATH)

**Entrega y límites de Bronze**

Operations conserva configuración, código, logs, eventos Spark, plan, recibos, checkpoints, manifiesto, bundle y marcador final. Bronze organiza los originales bajo `<fuente>/snapshot_id=<id>/`. Los buckets Silver, Gold y Artifacts no reciben escrituras de esta libreta.

El consumidor de Silver debe validar el `_SUCCESS` de corrida referenciado por el bundle y los marcadores de sus cinco snapshots. La existencia aislada de archivos, un plan o un checkpoint no acredita una ingesta completa. Las fechas de observación desconocidas permanecen nulas; la incorporación a Bronze registra su instante real.

La libreta no ejecuta limpieza, integración, EDA ni entrenamiento. Esas etapas consumen la publicación completa de Bronze. Las pruebas locales comprueban el comportamiento del código; el éxito nacional se acredita exclusivamente mediante la ejecución de esta libreta en EMR.

**Cierre de la sesión**

El cierre libera los executors de YARN y conserva la publicación S3. Para una nueva corrida se reinicia el kernel.


In [ ]:
spark.stop()
persist_logs(storage, settings, RUN_ID, LOG_PATH)
storage.close()
print("Sesión cerrada; la publicación Bronze queda disponible para Silver.")
